<a href="https://colab.research.google.com/github/abegithub2024/abegithub2024/blob/main/NEX_GDDP_CMIP6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Extracting NEX-GDDP-CMIP6 projections**

In [ ]:
import ee
import geemap
!pip install xee
import xee
import xarray as xr
import pandas as pd



In [4]:
map = geemap.Map(basemap ='SATELLITE')
map

AttributeError: module 'ee.data' has no attribute '_credentials'

In [ ]:
!pip install earthengine-api --upgrade
!pip install geemap --upgrade

In [2]:
import ee
import geemap

# Trigger OAuth2 authentication flow
ee.Authenticate()
ee.Initialize(
    project = 'ee-abehegeno',
    opt_url='https://earthengine-highvolume.googleapis.com')


In [3]:
# STEP 2: Define bounding box & sample grid cells
# ----------------------------------
LonMin, LonMax = 35.554494, 36.080954
LatMin, LatMax = -1.286116, -0.405454
geometry = ee.Geometry.Rectangle([LonMin, LatMin, LonMax, LatMax])

In [4]:
# Load one tasmax image to sample grid (25 km)
sample_img = ee.ImageCollection('NASA/GDDP-CMIP6') \
    .filterDate('2039-01-01', '2039-1-31') \
    .filter(ee.Filter.eq('model', 'ACCESS-ESM1-5')) \
    .filter(ee.Filter.eq('scenario', 'ssp245')) \
    .select('tasmax') \
    .first() \
    .clip(geometry)

In [5]:
# Sample the 6 grid cell centers as points
projection = sample_img.projection()
pixels = sample_img.sample(
    region=geometry,
    scale=projection.nominalScale(),
    projection=projection,
    geometries=True
).limit(6)

In [ ]:
# STEP 3: Extract daily tasmax for 2060
# ----------------------------------
model = 'ACCESS-ESM1-5'
scenario = 'ssp245'
year = 2039

In [ ]:
# Load full daily tasmax ImageCollection for 2060
collection = ee.ImageCollection('NASA/GDDP-CMIP6') \
    .filterDate(f'{year}-01-01', f'{year}-12-31') \
    .filter(ee.Filter.eq('model', model)) \
    .filter(ee.Filter.eq('scenario', scenario)) \
    .select('tasmax')


In [ ]:
# Extract tasmax for each grid point for each day
def extract_daily(image):
    date = ee.Date(image.get('system:time_start')).format('YYYY-MM-dd')
    sampled = image.reduceRegions(
        collection=pixels,
        reducer=ee.Reducer.mean(),
        scale=projection.nominalScale()
    ).map(lambda f: f.set('date', date))
    return sampled

In [ ]:
# Apply the function to each image
daily_fc = collection.map(extract_daily).flatten()


In [ ]:
# STEP 4: Export to Google Drive
# ----------------------------------
task = ee.batch.Export.table.toDrive(
    collection=daily_fc,
    description='tasmax_2039_ssp245_6grids',
    folder='NEX-GDDP',
    fileFormat='CSV'
)
task.start()

In [ ]:
print("✅ Export task started for 2039 daily tasmax.")

✅ Export task started for 2038 daily tasmax.
